# Ejercicio: Triggers

Antes de ejecutar, abre una terminal y deja corriendo:

```
nc -lk 9999
```

Ejecuta las celdas en orden. No ejecutes `spark.stop()` hasta el final: después de eso hay que reiniciar el kernel.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split
import time

spark = SparkSession.builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
lineasDF = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .load()

palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

## Funciones

Cada micro-batch se guarda con la hora a la que se procesó.

In [ ]:
registro = []

def guardar(df, batch_id):
    filas = sorted((r["palabra"], r["count"]) for r in df.collect())
    registro.append((time.strftime("%H:%M:%S"), batch_id, filas))

def mostrar():
    for hora, batch_id, filas in list(registro):
        print(f"{hora} | Batch {batch_id}: {filas}")

## Consulta con trigger de 1 minuto

In [ ]:
registro.clear()
print("Inicio:", time.strftime("%H:%M:%S"))

q = cantidadDF.writeStream \
    .outputMode("complete") \
    .foreachBatch(guardar) \
    .trigger(processingTime="1 minute") \
    .start()

Envía varias frases por `nc` con menos de un minuto de separación entre ellas. Ejecuta `mostrar()` cada cierto tiempo y observa a qué hora aparece realmente el resultado.

In [ ]:
mostrar()

In [ ]:
q.stop()

## Conclusión

Con `processingTime="1 minute"` Spark acumula los datos que llegan durante el intervalo y los procesa juntos al cumplirse el minuto. El resultado no aparece al enviar cada frase, sino cuando se dispara el trigger. Sin trigger, cada micro-batch se ejecuta apenas llegan los datos.

## Cierre (ejecutar solo al terminar)

In [ ]:
spark.stop()